# Multi-View 3D Line Reconstruction

This notebook demonstrates **multi-view 3D line reconstruction** using
LIMAP integration with lineextraction's native detection & matching.

**Pipeline:**
1. Load multi-view images with known camera poses
2. Detect line segments in each view (LSD)
3. Extract LBD descriptors
4. Run LIMAP multi-view triangulation
5. Visualize in 3D with Rerun

**Requirements:**
- [CVG LIMAP](https://github.com/cvg/limap) — *optional*; the notebook falls back
  to pairwise stereo if LIMAP is not installed. Install with:
  `uv sync --extra limap` (see `docs/JUPYTER.md` for details).
  **Do not** use `pip install limap` — the PyPI package is unrelated.
- MDB dataset or ETH3D scenes

> See `demo_stereo_reconstruction.ipynb` for the simpler two-view case.

In [ ]:
import sys
from pathlib import Path

# Discover workspace root
_nb = Path.cwd()
for _p in [_nb, *_nb.parents]:
    if (_p / 'MODULE.bazel').exists():
        WS = _p
        break
else:
    WS = _nb

# Ensure our Python packages are importable
for _d in ['python', 'bazel-bin/libs/geometry/python',
           'bazel-bin/libs/lsd/python', 'bazel-bin/libs/lfd/python',
           'bazel-bin/libs/edge/python', 'bazel-bin/libs/imgproc/python',
           'bazel-bin/libs/algorithm/python',  'bazel-bin/libs/eval/python']:
    p = str(WS / _d)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import cv2
print(f'Workspace: {WS}')

In [ ]:
import le_geometry
import le_lsd
import le_lfd

from lsfm.data import TestImages
from lsfm.reconstruction import (
    StereoConfig,
    StereoResult,
    reconstruct_lines_stereo,
)

# Check LIMAP availability
try:
    from lsfm.reconstruction import reconstruct_lines_multiview
    from lsfm.limap_compat import (
        LsfmLineDetector,
        LsfmLineDescriptor,
        LsfmLineMatcher,
    )
    HAS_LIMAP = True
    print('LIMAP available — multi-view reconstruction enabled')
except ImportError:
    HAS_LIMAP = False
    print('LIMAP not installed — using pairwise stereo fallback')

## 2. Load Multi-View Data

We support two data sources:
- **MDB stereo pairs** (always available) — used as a 2-view baseline
- **ETH3D / Hypersim** multi-view scenes — used when available

Set `DATA_SOURCE` below to choose.

In [ ]:
DATA_SOURCE = 'mdb'  # 'mdb', 'eth3d', or 'hypersim'
SCENE = 'Adirondack'

ds = TestImages()
images = []
cameras_dict = []

if DATA_SOURCE == 'mdb':
    # Two-view from MDB
    img_l = cv2.imread(str(ds.get(f'MDB/MiddEval3-H/{SCENE}/im0.png')), cv2.IMREAD_GRAYSCALE)
    img_r = cv2.imread(str(ds.get(f'MDB/MiddEval3-H/{SCENE}/im1.png')), cv2.IMREAD_GRAYSCALE)
    cam_l, cam_r = ds.stereo_camera_pair(SCENE)
    calib = ds.stereo_calibration(SCENE)
    images = [img_l, img_r]

    # Build camera dicts for multi-view API
    K_l = np.array([[calib['focal_x'], 0, calib['offset_x']],
                    [0, calib['focal_y'], calib['offset_y']],
                    [0, 0, 1]])
    cameras_dict = [
        {'K': K_l, 'R': np.eye(3), 't': np.zeros(3),
         'width': img_l.shape[1], 'height': img_l.shape[0]},
        {'K': K_l, 'R': np.eye(3), 't': np.array([calib.get('baseline', 0), 0, 0]),
         'width': img_r.shape[1], 'height': img_r.shape[0]},
    ]

elif DATA_SOURCE in ('eth3d', 'hypersim'):
    import json
    ds_name = 'ETH3D' if DATA_SOURCE == 'eth3d' else 'Hypersim'
    scene_dir = Path(f'resources/datasets/{ds_name}/{SCENE}')
    cam_path = WS / scene_dir / 'cameras.json'
    assert cam_path.exists(), f'Run setup_{DATA_SOURCE}.sh first: {cam_path}'

    with open(cam_path) as f:
        meta = json.load(f)

    for frame in meta['frames'][:10]:  # Limit for demo
        img_path = WS / scene_dir / frame['image']
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            images.append(img)
            cameras_dict.append({
                'K': np.array(frame['K']),
                'R': np.array(frame['R']),
                't': np.array(frame['t']),
                'width': frame.get('width', img.shape[1]),
                'height': frame.get('height', img.shape[0]),
            })

print(f'Loaded {len(images)} images ({DATA_SOURCE}/{SCENE})')
for i, img in enumerate(images):
    print(f'  View {i}: {img.shape[1]}x{img.shape[0]}')

## 3. Per-View Line Detection

Run LSD on each view and compute LBD descriptors.

In [ ]:
import time

MIN_LENGTH = 25.0

all_segs = []      # list of lists of LineSegment
all_desc_lists = [] # list of FdLBD descriptor lists
all_gx = []
all_gy = []

t0 = time.perf_counter()
for idx, img in enumerate(images):
    det = le_lsd.LsdCC()
    det.detect(img)
    segs = [s for s in det.line_segments() if s.length > MIN_LENGTH]
    all_segs.append(segs)

    gx = det.image_data()[0].astype(np.float32)
    gy = det.image_data()[1].astype(np.float32)
    all_gx.append(gx)
    all_gy.append(gy)

    lbd = le_lfd.FdcLBD(gx, gy)
    desc_list = lbd.create_list(segs)
    all_desc_lists.append(desc_list)

    print(f'  View {idx}: {len(segs)} segments, {len(desc_list)} descriptors')

detection_time = time.perf_counter() - t0
print(f'Detection + descriptors: {detection_time:.3f}s')

## 4. Pairwise Matching

Match descriptors between all view pairs.
For stereo pairs this is just one pair; for multi-view we match
all consecutive pairs (and optionally wider baselines).

In [ ]:
from itertools import combinations

# Build view pairs (consecutive + one-skip for multi-view)
n_views = len(images)
if n_views == 2:
    view_pairs = [(0, 1)]
else:
    view_pairs = [(i, i+1) for i in range(n_views - 1)]
    # Add skip-1 pairs for redundancy
    view_pairs += [(i, i+2) for i in range(n_views - 2)]

pair_matches = {}  # (vi, vj) -> list of (idx_i, idx_j)

t0 = time.perf_counter()
for vi, vj in view_pairs:
    matcher = le_lfd.BruteForceLBD()

    # Use stereo filter for 2-view MDB pairs (use relaxed thresholds
    # matching StereoConfig defaults — MDB has wide disparity).
    if n_views == 2 and DATA_SOURCE == 'mdb':
        sf = le_lfd.StereoLineFilter(
            height=images[vi].shape[0], max_dis_px=10000.0,
            angle_th=30.0, min_y_overlap=0.3)
        sf.train(all_segs[vi], all_segs[vj])
        matcher.train_filtered_stereo(all_desc_lists[vi], all_desc_lists[vj], sf)
    else:
        matcher.train(all_desc_lists[vi], all_desc_lists[vj])

    best = matcher.best()

    # Rotation consistency filter
    rf = le_lfd.GlobalRotationFilter()
    rf.train(all_segs[vi], all_segs[vj])

    pairs = []
    for i, m in enumerate(best):
        if np.isnan(m.distance):
            continue
        if not rf.filter(i, m.match_idx):
            continue
        pairs.append((i, m.match_idx))

    pair_matches[(vi, vj)] = pairs
    print(f'  Pair ({vi},{vj}): {len(pairs)} matches')

matching_time = time.perf_counter() - t0
print(f'Matching: {matching_time:.3f}s')

## 5. 3D Reconstruction

**Path A (LIMAP available):** Use multi-view triangulation with
track building and bundle adjustment.

**Path B (fallback):** Triangulate each view pair independently,
then merge colinear 3D segments.

In [ ]:
from lsfm.reconstruction import _merge_colinear_3d, _segments_to_f64

if HAS_LIMAP and n_views > 2:
    # --- Path A: LIMAP multi-view ---
    print('Using LIMAP multi-view reconstruction...')
    t0 = time.perf_counter()
    mv_result = reconstruct_lines_multiview(
        images, cameras_dict,
    )
    recon_time = time.perf_counter() - t0
    lines_3d_raw = mv_result['lines_3d']
    print(f'LIMAP: {mv_result["n_lines_3d"]} 3D lines in {recon_time:.3f}s')

    # Convert to LineSegment3 for consistent API
    segments_3d = []
    for arr in lines_3d_raw:
        if len(arr) >= 6:
            seg = le_geometry.LineSegment3.from_endpoints(
                float(arr[0]), float(arr[1]), float(arr[2]),
                float(arr[3]), float(arr[4]), float(arr[5]))
            segments_3d.append(seg)
else:
    # --- Path B: Pairwise stereo fallback ---
    if n_views == 2 and DATA_SOURCE == 'mdb':
        print('Using stereo reconstruction (2-view MDB)...')
        result = reconstruct_lines_stereo(
            images[0], images[1], cam_l, cam_r,
            config=StereoConfig(merge_enabled=True))
        segments_3d = result.segments_3d
        recon_time = sum(result.timings.values())
        print(f'Stereo: {result.n_lines_3d} 3D lines in {recon_time:.3f}s')
    else:
        print('Using pairwise triangulation fallback...')
        t0 = time.perf_counter()
        all_pair_3d = []
        for (vi, vj), matches in pair_matches.items():
            if not matches:
                continue
            # Build cameras from dicts
            Kl = cameras_dict[vi]['K']
            Kr = cameras_dict[vj]['K']
            cl = le_geometry.Camera_f64(
                focal_x=Kl[0][0], focal_y=Kl[1][1],
                offset_x=Kl[0][2], offset_y=Kl[1][2])
            cr = le_geometry.Camera_f64(
                focal_x=Kr[0][0], focal_y=Kr[1][1],
                offset_x=Kr[0][2], offset_y=Kr[1][2],
                tx=cameras_dict[vj]['t'][0] - cameras_dict[vi]['t'][0])

            stereo = le_geometry.Stereo_f64(cl, cr)
            ml = _segments_to_f64([all_segs[vi][a] for a, _ in matches])
            mr = _segments_to_f64([all_segs[vj][b] for _, b in matches])
            segs3d = stereo.triangulate_segments(ml, mr)
            all_pair_3d.extend(segs3d)

        # Merge across pairs
        segments_3d = _merge_colinear_3d(all_pair_3d, angle_th=5.0, dist_th=1.0)
        recon_time = time.perf_counter() - t0
        print(f'Pairwise: {len(all_pair_3d)} raw → {len(segments_3d)} merged 3D lines in {recon_time:.3f}s')

print(f'\nFinal: {len(segments_3d)} 3D line segments')

## 6. Statistics & Analysis

In [ ]:
if segments_3d:
    lengths = [s.length for s in segments_3d]
    print(f'3D Line Statistics:')
    print(f'  Count:  {len(segments_3d)}')
    print(f'  Length: min={min(lengths):.2f}  median={np.median(lengths):.2f}  max={max(lengths):.2f}')

    # Depth analysis (z-coordinate of segment midpoints)
    depths = []
    for seg in segments_3d:
        sp = np.array(seg.start_point())
        ep = np.array(seg.end_point())
        mid = (sp + ep) / 2
        depths.append(mid[2])
    depths = np.array(depths)
    print(f'  Depth:  min={depths.min():.2f}  median={np.median(depths):.2f}  max={depths.max():.2f}')

    # Per-view match counts
    print(f'\nPer-pair match counts:')
    for (vi, vj), matches in pair_matches.items():
        print(f'  ({vi},{vj}): {len(matches)} matches')

    # Timing summary
    print(f'\nTiming:')
    print(f'  Detection:      {detection_time:.3f}s')
    print(f'  Matching:       {matching_time:.3f}s')
    print(f'  Reconstruction: {recon_time:.3f}s')
    print(f'  Total:          {detection_time + matching_time + recon_time:.3f}s')
else:
    print('No 3D segments reconstructed.')

## 7. Rerun 3D Visualization

Visualize the multi-view reconstruction with Rerun:
- Camera frustums for all views
- Detected 2D segments per view
- Reconstructed 3D lines

In [ ]:
try:
    import rerun as rr
    import rerun.blueprint as rrb
    HAS_RERUN = True
except ImportError:
    HAS_RERUN = False
    print('Rerun not installed — skipping visualization')

In [ ]:
if HAS_RERUN:
    from scipy.spatial.transform import Rotation as ScipyR

    rr.init('multiview_reconstruction', spawn=False)

    # Helper: segments to strips
    def segs_to_strips(segs):
        pts, colors = [], []
        for s in segs:
            ep = s.end_points()
            pts.extend([[ep[0], ep[1]], [ep[2], ep[3]]])
            colors.extend([[0, 200, 100, 255]] * 2)
        return np.array(pts), np.array(colors, dtype=np.uint8)

    def segs3d_to_strips(segs):
        pts, colors = [], []
        for s in segs:
            sp, ep = s.start_point(), s.end_point()
            pts.extend([list(sp), list(ep)])
            colors.extend([[255, 80, 40, 255]] * 2)
        return np.array(pts), np.array(colors, dtype=np.uint8)

    # Log per-view data
    for idx in range(n_views):
        # Camera frustum
        K = np.array(cameras_dict[idx]['K'])
        R = np.array(cameras_dict[idx].get('R', np.eye(3)))
        t = np.array(cameras_dict[idx].get('t', np.zeros(3)))
        w = cameras_dict[idx].get('width', images[idx].shape[1])
        h = cameras_dict[idx].get('height', images[idx].shape[0])

        # Camera-to-world: R_cw = R^T, t_cw = -R^T @ t
        R_cw = R.T
        t_cw = -R_cw @ t
        quat_xyzw = ScipyR.from_matrix(R_cw).as_quat()  # [x, y, z, w]

        rr.log(f'world/cameras/view_{idx}',
               rr.Pinhole(image_from_camera=K, width=w, height=h))
        rr.log(f'world/cameras/view_{idx}',
               rr.Transform3D(
                   rotation=rr.datatypes.Quaternion(
                       xyzw=quat_xyzw),
                   translation=t_cw))
        rr.log(f'world/cameras/view_{idx}/image',
               rr.Image(images[idx]))

        # 2D segments overlay
        if all_segs[idx]:
            pts, cols = segs_to_strips(all_segs[idx])
            rr.log(f'world/cameras/view_{idx}/lines_2d',
                   rr.LineStrips2D(pts.reshape(-1, 2, 2), colors=cols[::2]))

    # Log 3D lines
    if segments_3d:
        pts3, cols3 = segs3d_to_strips(segments_3d)
        rr.log('world/lines_3d',
               rr.LineStrips3D(pts3.reshape(-1, 2, 3), colors=cols3[::2]))

    # Log matches as connections
    for (vi, vj), matches in pair_matches.items():
        if not matches:
            continue
        rr.log(f'world/matches/{vi}_{vj}',
               rr.TextLog(f'{len(matches)} matches between view {vi} and {vj}'))

    # Blueprint
    bp = rrb.Blueprint(
        rrb.Horizontal(
            rrb.Spatial3DView(name='3D Scene', origin='world'),
            rrb.Vertical(
                *[rrb.Spatial2DView(name=f'View {i}',
                                   origin=f'world/cameras/view_{i}')
                  for i in range(min(n_views, 4))],
            ),
            column_shares=[2, 1],
        ),
    )

    rr.notebook_show(blueprint=bp)
    print(f'Rerun: {n_views} cameras + {len(segments_3d)} 3D lines')

## 8. Comparison: Stereo vs. Multi-View

If we have more than 2 views, compare the pairwise stereo result
with the LIMAP multi-view result.

In [ ]:
if n_views >= 2 and DATA_SOURCE == 'mdb':
    # Run stereo baseline for comparison
    stereo_result = reconstruct_lines_stereo(
        images[0], images[1], cam_l, cam_r,
        config=StereoConfig())

    print('=== Stereo (2-view) ===')
    print(f'  3D lines: {stereo_result.n_lines_3d}')
    print(f'  Timings:  {stereo_result.timings}')
    if stereo_result.reproj_errors.size > 0:
        print(f'  Reproj error: median={np.median(stereo_result.reproj_errors):.2f} px')

    print(f'\n=== Multi-view / Fallback ===')
    print(f'  3D lines: {len(segments_3d)}')

    if HAS_LIMAP and n_views > 2:
        print(f'  Method:   LIMAP multi-view')
    else:
        print(f'  Method:   Pairwise fallback')
else:
    print('Comparison requires MDB data (set DATA_SOURCE="mdb").')

## Summary

This notebook demonstrated:

| Feature | Details |
|---------|--------|
| **Detection** | LSD + length filter per view |
| **Descriptors** | LBD via FdcLBD |
| **Matching** | BruteForceLBD + stereo/rotation filters |
| **Reconstruction** | LIMAP multi-view *or* pairwise stereo fallback |
| **Visualization** | Rerun 3D scene with camera frustums |

### Next Steps
- Run `setup_eth3d.sh` or `setup_hypersim.sh` for real multi-view scenes
- Install LIMAP for proper multi-view triangulation with track building
- See `evaluation/python/tools/reconstruction_benchmark.py` for quantitative evaluation